# 🟢 LangSmith — Tier 1: Error path

Same scenario as the Langfuse version, traced through LangSmith. The agent asks about `Nobody McNobody` and `get_ads_id` returns an error string.

**What to look for in the LangSmith UI**:
- Project from `LANGSMITH_PROJECT` env var → Runs filtered by tag `tier1` + `error_path`
- The failing tool call shows up as a tool-type Run with the ERROR string in its output
- LangSmith's UI does NOT mark this as an error span automatically (the tool returned a string, not an exception); the agent's recovery is visible in the next LLM run
- Click the trace → see the playground button on any LLM span for replay

In [ ]:
import os, sys, pathlib
from dotenv import load_dotenv

ROOT = pathlib.Path().resolve().parent.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
load_dotenv(ROOT / ".env")

os.environ["LANGSMITH_TRACING"] = "true"
os.environ.setdefault("LANGSMITH_ENDPOINT", "https://api.smith.langchain.com")
os.environ.setdefault("LANGSMITH_PROJECT", "observability-comparison")

from langchain_core.messages import HumanMessage
from shared.workflow import build_agent, SCENARIOS

scenario = next(s for s in SCENARIOS if s["id"] == "error_path")
print(f"Scenario: {scenario['id']}")
print(f"Prompt  : {scenario['prompt']}")
print(f"Tracing to LangSmith project: {os.environ['LANGSMITH_PROJECT']}")

In [ ]:
agent = build_agent(prompt_source="langsmith")

config = {
    "run_name": scenario["id"],
    "tags": ["tier1", "error_path", "langsmith"],
    "metadata": {
        "scenario_id": scenario["id"],
        "thread_id": "tier1-error-path-demo",
        "session_id": "tier1-error-path-demo",
    },
}

out = agent.invoke({"messages": [HumanMessage(content=scenario["prompt"])]}, config=config)

for i, m in enumerate(out["messages"]):
    role = type(m).__name__
    content = (m.content if isinstance(m.content, str) else str(m.content))[:200]
    tool_calls = getattr(m, "tool_calls", None) or []
    print(f"  [{i}] {role:<14} {content}")
    for tc in tool_calls:
        name = tc.get("name") if isinstance(tc, dict) else tc.name
        print(f"        tool_call -> {name}")

print("\nDone. Open https://smith.langchain.com -> your project -> filter tag `error_path`.")

## Deck takeaway

**LangSmith on errors**: tool Runs always show their output payload; an ERROR string surfaces in the output panel. LangSmith *does* distinguish between Runs with `status='error'` (raised exceptions) and Runs with `status='success'` whose output happens to contain an error message — ours is the latter, so the success/error UI distinction won't trigger.

If you'd configured an evaluator looking for `"ERROR:"` in tool outputs, that'd flag this trace. None of the three platforms makes that automatic — except Galileo's Tool Error Rate, which is keyed on the SDK-level success signal, not the payload.